# Does the Pareto Principle (80/20 Rule) Apply to my Streaming History?

In [28]:
# necessary imports

import pandas as pd
from utils import historySplit
import dotenv
import os
import requests

## Top Artists

Manually getting top artist data again, ensuring as much accuracy as possible due to rounding of the top artists csv

In [4]:
df = pd.read_csv("Spotify Extended Streaming History/cleaned/cleaned-history.csv")

In [5]:
allArtists = df['master_metadata_album_artist_name'].unique()
topArtists = pd.DataFrame({'Artists': allArtists})
topArtists['time_played'] = topArtists['Artists'].apply(lambda x: df[df['master_metadata_album_artist_name'] == x]['ms_played'].sum())
topArtists.sort_values(by='time_played', ascending=False, inplace=True)
topArtists.reset_index(drop=True, inplace=True)

In [6]:
n_rows = 0.2 * len(topArtists)
top20Percent = topArtists.head(int(n_rows))

In [8]:
twentyTimePlayed = top20Percent['time_played'].sum()
allTimePlayed = topArtists['time_played'].sum()


In [11]:
twentyPercentTimePlayed = (twentyTimePlayed / allTimePlayed) * 100
print(f"My top 20% of artists account for {twentyPercentTimePlayed:.2f}% of my total listening time.")

My top 20% of artists account for 94.27% of my total listening time.


My top artists actually exceed the Pareto principle's by a noticeable margin!

## Top Tracks

Favouring speed of processing, using the top tracks csv already saved as processing the results often takes excessive amounts of time

In [12]:
topTracks = pd.read_csv("Spotify Extended Streaming History/cleaned/top-tracks.csv")

In [13]:
n_tracks = 0.2 * len(topTracks)
top20PercentTracks = topTracks.head(int(n_tracks))

In [17]:
twentyPercentTracksPlayed = top20PercentTracks['time_played'].sum()
allTracksPlayed = topTracks['time_played'].sum()
twentyPercentTracksPlayedPercentage = (twentyPercentTracksPlayed / allTracksPlayed) * 100
print(f"My top 20% of tracks account for {twentyPercentTracksPlayedPercentage:.2f}% of my total listening time.")

My top 20% of tracks account for 82.98% of my total listening time.


The Pareto principle still holds for the top 20th percentile of tracks listened, but there is not as much of a bias as with my top artists.

## Top Genres

Using the pre-sorted top genres csv file to reduce API calls, the accuracy dropoff is negligible given the scale of the numbers in the genres csv.

In [18]:
topGenres = pd.read_csv("Spotify Extended Streaming History/cleaned/top-genres.csv")

In [19]:
n_genres = 0.2 * len(topGenres)
top20PercentGenres = topGenres.head(int(n_genres))

In [20]:
twentyPercentGenres = top20PercentGenres['time_played'].sum()
allGenresPlayed = topGenres['time_played'].sum()
twentyPercentGenresPlayedPercentage = (twentyPercentGenres / allGenresPlayed) * 100
print(f"My top 20% of genres account for {twentyPercentGenresPlayedPercentage:.2f}% of my total listening time.")

My top 20% of genres account for 67.38% of my total listening time.


The Pareto principle does not hold on my top genres listened! Partially this can be chalked up to the crowd-sourced genres on last.fm including many specific subgenres that will outweigh the top 20% (i.e. abstract hip-hop, jazz rap, math rock etc). However, the main cause that affects this is likely that each song is tagged with multiple genres. To fix this and see the real distribution of my genre listening, I will need to re-run the last.fm API calls and only count the primary genre of each song, as opposed to the top 3.

In [26]:
genreArtists = topArtists.head(150)

In [29]:
# get genre for each artist using Last.fm API
dotenv.load_dotenv()
LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")

artist_genre_lookup = {}

url = "http://ws.audioscrobbler.com/2.0/"

# Iterate through each artist and fetch their top tags (genres) from Last.fm
for index, artist_name in enumerate(genreArtists['Artists']):
    try:
        # Prepare the payload for the Last.fm API request
        payload = {
            'method': 'artist.getTopTags',
            'artist': artist_name,
            'api_key': LASTFM_API_KEY,
            'format': 'json'
        }
        
        # Make the API request to Last.fm
        response = requests.get(url, params=payload)
        data = response.json()
        
        # Extract the top tags (genres) from the response
        toptags_container = data.get('toptags', {})
        raw_tags_list = toptags_container.get('tag', [])
        
        # Extract the top genre (if available) and convert them to lowercase
        genres = [tag['name'].lower() for tag in raw_tags_list[:1]] # get top genre
        
        
        if not genres:
            genres = ['unknown']
            
        artist_genre_lookup[artist_name] = genres
    
    # Handle specific exceptions
    except Exception as e:
        print(f"Skipping {artist_name} due to an unexpected error: {e}")
        continue

In [ ]:
genreArtists['genres'] = genreArtists['Artists'].map(artist_genre_lookup)
genreArtists['genres'] = genreArtists['genres'].fillna({i: ['unknown'] for i in genreArtists.index})
genreArtists = genreArtists.explode('genres', ignore_index=True)
genre_sum = genreArtists.groupby('genres')['time_played'].sum().reset_index()
genre_sum.sort_values(by='time_played', ascending=False, inplace=True)
genre_sum.reset_index(drop=True, inplace=True)

,Artists,Genres
0,Gorillaz,[electronic]
1,Radiohead,[rock]
2,Elliott Smith,[singer-songwriter]
3,King Gizzard & The Lizard Wizard,[psychedelic rock]
4,The Beatles,[classic rock]
